## Imports

In [ ]:
import os
from dotenv import load_dotenv
from tqdm.notebook import tqdm
from datasets import load_dataset

load_dotenv(override=True)

## Load our Datasets

In [ ]:
# Needs the repo's loading script, so datasets is pinned to 3.6.0 (4.0 dropped script support).
# Note: streaming=True fails here — pyarrow can't do the cast the script's schema implies.

dataset = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    "raw_meta_Appliances",
    split="full",
    trust_remote_code=True,
)

## Find the most Expensive one.

In [ ]:
# The script stores price as a string, using "None" for missing, so parse before comparing.


def to_price(value):
    try:
        return float(value)
    except TypeError, ValueError:
        return None


prices = dataset[
    "price"
]  # materializes the column; the slow part, before tqdm sees anything
priced = [
    (price, i)
    for i, value in tqdm(enumerate(prices), total=len(prices))
    if (price := to_price(value)) is not None
]
priced.sort(reverse=True)

print(f"{len(priced):,} of {len(dataset):,} items have a price\n")
for price, i in priced[:10]:
    print(f"${price:>10,.2f}  {dataset[i]['title'][:60]}")

most_expensive = dataset[priced[0][1]]
print(f"\nMost expensive: {most_expensive['title']}")
print(f"Price:  ${float(most_expensive['price']):,.2f}")
print(f"Store:  {most_expensive['store']}")
print(
    f"Rating: {most_expensive['average_rating']} from {most_expensive['rating_number']:,} ratings"
)